In [28]:
import os
import pandas as pd
import numpy as np
from pathlib import Path
from utils.graph_features import network_creation, features_generation
import pyarrow.parquet as pq
import pyarrow as pa

PROJECT_ROOT = Path.cwd().resolve().parent.parent
DATA_EXPLODED_PATH = PROJECT_ROOT / 'data' / 'exploded_splits'

TRAIN_PATH = DATA_EXPLODED_PATH / "train_pairs.parquet"
VAL_PATH = DATA_EXPLODED_PATH / "validation_pairs.parquet"
TEST_PATH = DATA_EXPLODED_PATH / "test_pairs.parquet"

OUTPUT_DIR =  PROJECT_ROOT / 'data' / 'graph_features'

In [15]:
# load the file 
train = pd.read_parquet(TRAIN_PATH)
val = pd.read_parquet(VAL_PATH)
test = pd.read_parquet(TEST_PATH)

# Feature engineering: normal features

## 1. Exploration
Briefly exploration of the columns and the type.

In [12]:
train.head(2)

,article_id,ref_id,title_article,lang_article,doc_type_article,abstract_article,year_article,n_citation_article,keywords_article,authors_article,title_ref,lang_ref,doc_type_ref,abstract_ref,year_ref,keywords_ref,authors_ref,is_reference_valid,split
0,53e99f0ab7602d97027d6a89,53e99ff0b7602d97028d14d3,Comments on a Method of Karpovsky,en,journal,Karpovsky has presented methods of multiple-va...,1978.0,12.0,"[karpovsky, method]","[{'id': '53f44aa3dabfaee43ec8ec34', 'name': 'C...",Harmonic-Analysis over Finite Commutative Grou...,en,journal,In this paper we consider the linearization pr...,1977.0,None,"[{'id': '562c496445cedb3398bde732', 'name': 'M...",1,train
1,53e9bd81b7602d9704a24d06,557f4d4f6fee0fe990cb035f,The Effect of Visual Information on Word Initi...,en,conference,Disabled individuals can realize many benefits...,1996.0,5.0,"[audio-visual systems, handicapped aids, heari...","[{'id': '53f44a07dabfaeb22f4cee3a', 'name': 'R...",Improving Connected Letter Recognition by Lipr...,en,conference,In this paper we show how recognition performa...,1993.0,"[error rate, connected word recognition proble...","[{'id': '5484d790dabfae9b40133197', 'name': 'C...",1,train


## 2. Filter
First of all not all the features are usefull for our project, we should select the ones that can be used.


In [16]:
train.columns

Index(['article_id', 'ref_id', 'title_article', 'lang_article',
       'doc_type_article', 'abstract_article', 'year_article',
       'n_citation_article', 'keywords_article', 'authors_article',
       'title_ref', 'lang_ref', 'doc_type_ref', 'abstract_ref', 'year_ref',
       'keywords_ref', 'authors_ref', 'is_reference_valid', 'split'],
      dtype='str')

In [17]:
cols_to_exclude = ['article_id', 'ref_id', 'title_article', 'abstract_article', 'title_ref', 'abstract_ref', 'split']

for df in [train, val, test]:
    df.drop(cols_to_exclude, inplace=True, axis=1)

In [18]:
train.columns

Index(['lang_article', 'doc_type_article', 'year_article',
       'n_citation_article', 'keywords_article', 'authors_article', 'lang_ref',
       'doc_type_ref', 'year_ref', 'keywords_ref', 'authors_ref',
       'is_reference_valid'],
      dtype='str')

## 3. Feature Engineering
Since usually the models cannot use categorical or string columns, the column should be converted in a model readable format.

In [19]:
display(train.dtypes)

lang_article              str
doc_type_article          str
year_article          float64
n_citation_article    float64
keywords_article       object
authors_article        object
lang_ref                  str
doc_type_ref              str
year_ref              float64
keywords_ref           object
authors_ref            object
is_reference_valid      int64
dtype: object

### 3.1 Add some features
Before modify, add some useful columns regarding the columns that we are going to modify.

Since keywords and authors are strctured data as np.ndarrays, before embed them we will create the columns that count how many of them are in the paper.

In [52]:
for df in [train, val, test]:
    # add n_keywords_article and n_keywords_ref
    # add n_authors_article and n_authors_ref
    for col_to_sum in ['keywords_article', 'keywords_ref', 'authors_article', 'authors_ref']:
        counter_series = df[col_to_sum].apply(lambda obj: 0 if not isinstance(obj, np.ndarray) else len(obj))
        df['n_'+col_to_sum] = counter_series

In [57]:
train[['keywords_article', 'keywords_ref', 'n_keywords_article', 'n_keywords_ref', 'authors_article', 'authors_ref', 'n_authors_article', 'n_authors_ref']].head(2)

,keywords_article,keywords_ref,n_keywords_article,n_keywords_ref,authors_article,authors_ref,n_authors_article,n_authors_ref
0,"[karpovsky, method]",None,2,0,"[{'id': '53f44aa3dabfaee43ec8ec34', 'name': 'C...","[{'id': '562c496445cedb3398bde732', 'name': 'M...",1,1
1,"[audio-visual systems, handicapped aids, heari...","[error rate, connected word recognition proble...",23,11,"[{'id': '53f44a07dabfaeb22f4cee3a', 'name': 'R...","[{'id': '5484d790dabfae9b40133197', 'name': 'C...",2,4


In [ ]:
# encode only most popular 10 keywords, lang, doc type

### 3.2 Encoding not numeric features
For simple features that have few variant, we could opt to encode with a one hot encoding.
But for others we could need to opt for other choices, for exapmple for the keywords we would one hot encode only the most frequent, for authors, we will ...